In [1]:
from pathlib import Path
import os 

import pandas as pd
import anndata as ad 

from dotenv import dotenv_values

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

ADATA_FP =  DATADIR / "9_objects" / "singlecell" / "hypomap3d_singlecell_all.h5ad"
QC_FP = DATADIR / "9_objects" / "hypomap3d_singlecell_QC_log.csv"

In [ ]:
adata = ad.read_h5ad(ADATA_FP, backed="r")
qc = pd.read_csv(QC_FP)

# During analysis we realized that there were 3707 missing cells from SC1-1 that 
# did not fail QC but were not part of the output created after QC filtering. 
# It is currently assumed that this was due to data corruption during the saving process.
# Regardless, these cells must be treated as "failed_qc_thresholds"
sc1 = ad.read_h5ad(DATADIR / "2b_clean" / "SC1-01.h5ad", backed="r")
n_missing = len(qc.loc[(qc.Dataset == "SC1-01") & (~ (qc.failed_qc_thresholds | qc.qc_manually_removed))]) - len(sc1)
n_missing

3707

In [ ]:
# copy and pasted from version v0.6
# TO DO: write code to produce this given the updated directory structure
pre_cb_cells = pd.read_csv(DATADIR / "publication" / "Ext_Fig_01" / "cell_flow_pre_cellbender.csv")

In [39]:
pre_cb_cells.head(2)

,dataset,sample,num cells,step
0,SC2-13,M2TX_200924_203_C01,3940799,pre_cellbender
1,SC1-03,SAMN06215610,3961584,pre_cellbender


In [ ]:
n_pre_cb = pre_cb_cells["num cells"].sum()
n_non_empty_droplets = len(qc) #post cellbender
n_empty_droplets = n_pre_cb - n_non_empty_droplets

n_failed_qc = (qc.failed_qc_thresholds | qc.qc_manually_removed).sum()
n_passed_qc = n_non_empty_droplets - n_failed_qc - n_missing 

n_doublets_removed = (qc.IsRemovedDoublet | qc.IsSoloDoublet | qc.IsDoubletCluster | qc.IsDoubletManual).sum() # manual removal was done in non-hy removal notebook, before feeding into clustering
n_non_doublets = n_passed_qc - n_doublets_removed

n_nonhy_removed = (qc.IsNonHy | qc.IsNonHyManual | qc.IsMajorFeatureLowConfidence).sum() # major features is pre clustering
n_hy = n_non_doublets - n_nonhy_removed

n_outliers = qc.IsSingletonCluster.sum() # done in non-hy notebook
n_into_clustering = n_hy - n_outliers 

n_post_hoc_removed = (qc.IsClusteringRemoved | qc.IsPosthocRemoved | qc.IsLastMinuteRemoved).sum() #Notebooks: 5_clustering/Recombine, 5_clustering/posthoc_qc; 9_objects/1a_singlecell
n_final = n_into_clustering - n_post_hoc_removed

print(f"During alignment, {n_pre_cb} cells were called.")
print(f"Via cellbender, {n_empty_droplets} empty droplets were removed, yielding {n_non_empty_droplets} cells.")
print(f"{n_failed_qc} cells were removed via QC thresholds, yielding {n_passed_qc} cells.")
print(f"Those cells were fed into doublet detection, in which {n_doublets_removed} cells were removed, yielding {n_non_doublets} cells.")
print(f"{n_nonhy_removed} cells were removed as non-hypothalamic, yielding {n_hy} HY cells.")
print(f"{n_outliers} cells were removed as outliers (singleton clusters) after which {n_into_clustering} cells were fed into the clustering (but only the control cells)")
print(f"Lastly, {n_post_hoc_removed} cells were removed posthoc, yielding the final singlecell object with {n_final} cells")

assert n_final == len(adata)

During alignment, 581586653 cells were called.
Via cellbender, 577491287 empty droplets were removed, yielding 4095366 cells.
2060889 cells were removed via QC thresholds, yielding 2030770 cells.
Those cells were fed into doublet detection, in which 307994 cells were removed, yielding 1722776 cells.
132785 cells were removed as non-hypothalamic, yielding 1589991 HY cells.
359 cells were removed as outliers (singleton clusters) after which 1589632 cells were fed into the clustering (but only the control cells)
Lastly, 25568 cells were removed posthoc, yielding the final singlecell object with 1564064 cells
